Check: The hemoglobin concentration stratified by both antepartum and postpartum hemorrhage incidence (also stratified by anemia status in pregnancy to avoid confounding by this factor) should differ along each axis by the magnitude of the relevant hemoglobin shift, in each post-pregnancy time step.

In [1]:
from vivarium import Artifact, InteractiveContext
import pandas as pd, numpy as np, os

In [2]:
! pip list | grep vivarium

# [software verion + hash . date]

vivarium                                 4.0.3
vivarium_build_utils                     3.2.2
vivarium_cluster_tools                   3.1.6
vivarium-compat                          0.6.1
vivarium-dependencies                    1.1.0
vivarium_gates_mncnh                     34.1.dev8+g5eaecaf6f.d20260616 /mnt/share/homes/tylerdy/vivarium_gates_mncnh
vivarium-gbd-mapping                     6.0.1
vivarium_public_health                   5.0.2
vivarium-risk-distributions              3.1.1
vivarium_testing_utils                   0.5.5


In [3]:
! pip freeze | grep vivarium

vivarium==4.0.3
vivarium-compat==0.6.1
vivarium-dependencies==1.1.0
vivarium-gbd-mapping==6.0.1
vivarium-risk-distributions==3.1.1
vivarium_build_utils==3.2.2
vivarium_cluster_tools==3.1.6
-e git+https://github.com/ihmeuw/vivarium_gates_mncnh.git@316c7cf9cfbc1c5bec0178c2a5ca9360eccd42fa#egg=vivarium_gates_mncnh
vivarium_public_health==5.0.2
vivarium_testing_utils==0.5.5


In [4]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from vivarium import InteractiveContext, Artifact

In [ ]:
import vivarium_gates_mncnh
from vivarium.framework.configuration import build_model_specification
from pathlib import Path

from vivarium_gates_mncnh.constants.data_values import (
    SIMULATION_EVENT_NAMES,
    COLUMNS,
    PREGNANCY_OUTCOMES,
    PIPELINES,
)

path = Path(vivarium_gates_mncnh.__file__).parent / 'model_specifications/model_spec.yaml'
custom_model_specification = build_model_specification(path)
custom_model_specification.configuration.input_data.input_draw_number = 60

custom_model_specification.configuration.population.population_size = 20_000 * 3

included_components = custom_model_specification.components.vivarium_gates_mncnh.components
included_components

['Hemoglobin()',
 'AnemiaInterventionPropensity()',
 'AgelessPopulation("population.scaling_factor")',
 'Pregnancy()',
 'ResultsStratifier()',
 'ANCAttendance()',
 'Ultrasound()',
 'MaternalDisorder("maternal_obstructed_labor_and_uterine_rupture")',
 'MaternalDisorder("maternal_sepsis_and_other_maternal_infections")',
 'AntepartumHemorrhage()',
 'PostpartumHemorrhage()',
 'ResidualMaternalDisorders()',
 'AbortionMiscarriageEctopicPregnancy()',
 'MaternalDisordersBurden()',
 'LBWSGRisk()',
 "LBWSGRiskEffect('cause.all_causes.all_cause_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_sepsis_and_other_neonatal_infections.cause_specific_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_preterm_birth_with_rds.cause_specific_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_preterm_birth_without_rds.cause_specific_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_encephalopathy_due_to_birth_asphyxia_and_trauma.cause_specific_mortality_risk')",
 "PretermBirth('neonatal_preterm_bi

In [6]:
artifact_path = custom_model_specification.configuration.input_data.artifact_path
art = Artifact(artifact_path)

artifact_path

'/mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf'

In [7]:
draw_num = custom_model_specification.configuration.input_data.input_draw_number
draw = 'draw_' + str(draw_num)
draw

'draw_60'

In [23]:
aph_on_hemoglobin_shift_early = art.load("risk_effect.hemorrhage_hemoglobin_shift.aph_shift_0_6w")[draw][0]
pph_on_hemoglobin_shift_early = art.load("risk_effect.hemorrhage_hemoglobin_shift.pph_shift_0_6w")[draw][0]
aph_on_hemoglobin_shift_late = art.load("risk_effect.hemorrhage_hemoglobin_shift.aph_shift_6w_9m")[draw][0]
pph_on_hemoglobin_shift_late = art.load("risk_effect.hemorrhage_hemoglobin_shift.pph_shift_6w_9m")[draw][0]
art_shifts = {
    "aph_on_hemoglobin_shift_early": aph_on_hemoglobin_shift_early, 
    "pph_on_hemoglobin_shift_early": pph_on_hemoglobin_shift_early, 
    "aph_on_hemoglobin_shift_late": aph_on_hemoglobin_shift_late, 
    "pph_on_hemoglobin_shift_late": pph_on_hemoglobin_shift_late,
}
art_shifts

{'aph_on_hemoglobin_shift_early': -9.319741765293287,
 'pph_on_hemoglobin_shift_early': -16.202410730463864,
 'aph_on_hemoglobin_shift_late': -5.875628017090329,
 'pph_on_hemoglobin_shift_late': -0.6569587720997878}

In [64]:
def check_hemoglobin_shift(sim: InteractiveContext):
    pop = sim.get_population([
        COLUMNS.ANTEPARTUM_HEMORRHAGE, 
        COLUMNS.POSTPARTUM_HEMORRHAGE, 
        COLUMNS.PREGNANCY_OUTCOME,
        COLUMNS.ANEMIA_STATUS_DURING_PREGNANCY,
        ])
    # Check full term with anemia status
    full_term = pop.loc[
        ((pop[COLUMNS.PREGNANCY_OUTCOME] == PREGNANCY_OUTCOMES.FULL_TERM_OUTCOME) |
        (pop[COLUMNS.PREGNANCY_OUTCOME] == PREGNANCY_OUTCOMES.LIVE_BIRTH_OUTCOME) |
        (pop[COLUMNS.PREGNANCY_OUTCOME] == PREGNANCY_OUTCOMES.STILLBIRTH_OUTCOME)) &
        pop[COLUMNS.ANEMIA_STATUS_DURING_PREGNANCY]
    ]
    has_aph = full_term[COLUMNS.ANTEPARTUM_HEMORRHAGE]
    has_pph = full_term[COLUMNS.POSTPARTUM_HEMORRHAGE]
    if len(full_term) > 0 and has_aph.sum() > 0 and has_pph.sum() > 0:
        # Get hemoglobin exposure values from the pipeline
        hgb = sim.get_population(PIPELINES.HEMOGLOBIN_EXPOSURE)
        anemia_statuses = full_term[COLUMNS.ANEMIA_STATUS_DURING_PREGNANCY].unique()
        hgb_means = {}
        hgb_means["next_step"] = sim._clock.step_name
        hgb_means["no_aph_no_pph"] = hgb.loc[full_term.index[~has_aph & ~has_pph]].mean()
        hgb_means["aph_no_pph"] = hgb.loc[full_term.index[has_aph & ~has_pph]].mean()
        hgb_means["no_aph_pph"] = hgb.loc[full_term.index[~has_aph & has_pph]].mean()
        hgb_means["aph_pph"] = hgb.loc[full_term.index[has_aph & has_pph]].mean()
        for anemia_status in anemia_statuses:
            has_status = full_term[COLUMNS.ANEMIA_STATUS_DURING_PREGNANCY] == anemia_status
            hgb_means[anemia_status] = hgb.loc[full_term.index[has_status]].mean()
            hgb_means["next_step"] = sim._clock.step_name
            hgb_means[f"{anemia_status}_no_aph_no_pph"] = hgb.loc[full_term.index[~has_aph & ~has_pph]].mean()
            hgb_means[f"{anemia_status}_aph_no_pph"] = hgb.loc[full_term.index[has_aph & ~has_pph]].mean()
            hgb_means[f"{anemia_status}_no_aph_pph"] = hgb.loc[full_term.index[~has_aph & has_pph]].mean()
            hgb_means[f"{anemia_status}_aph_pph"] = hgb.loc[full_term.index[has_aph & has_pph]].mean()
        return hgb_means

    # TODO check for age groups as well?

In [10]:
def run_all_steps(sim: InteractiveContext):
    step_shifts = []
    while sim.get_number_of_steps_remaining() > 0:
        step_shift = check_hemoglobin_shift(sim)
        if step_shift:
            step_shifts.append(step_shift)
        sim.step()
    return pd.DataFrame(step_shifts)

In [65]:
sim = InteractiveContext(custom_model_specification)
step_shifts = run_all_steps(sim)

2026-07-15 13:07:16.175 | INFO     | simulation_11-artifact_manager:80 - Running simulation from artifact located at /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf.
2026-07-15 13:07:16.177 | INFO     | simulation_11-artifact_manager:81 - Artifact base filter terms are ['draw == 60'].
2026-07-15 13:07:16.178 | INFO     | simulation_11-artifact_manager:82 - Artifact additional filter terms are None.


2026-07-15 13:07:55.237 | WARNING  | simulation_11-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-07-15 13:07:55.238 | WARNING  | simulation_11-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-07-15 13:07:55.410 | WARNING  | simulation_11-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure' during setup.
2026-07-15 13:07:55.411 | WARNING  | simulation_11-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'ensemble_distribution_weights' during setup.
2026-07-15 13:07:55.412 | WARNING  | simulation_11-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure_standard_deviation' during setup.
2026-07-15 13:07:55.413 | WARNING  | simulation_11-lookup_table_manager

In [66]:
step_shifts

,next_step,no_aph_no_pph,aph_no_pph,no_aph_pph,aph_pph,not_anemic,not_anemic_no_aph_no_pph,not_anemic_aph_no_pph,not_anemic_no_aph_pph,not_anemic_aph_pph,...,mild,mild_no_aph_no_pph,mild_aph_no_pph,mild_no_aph_pph,mild_aph_pph,severe,severe_no_aph_no_pph,severe_aph_no_pph,severe_no_aph_pph,severe_aph_pph
0,maternal_sepsis_and_other_maternal_infections,122.334312,112.490701,112.235207,92.380021,133.245203,122.334312,112.490701,112.235207,92.380021,...,108.057420,122.334312,112.490701,112.235207,92.380021,57.894915,122.334312,112.490701,112.235207,92.380021
1,residual_maternal_disorders,122.334312,112.490701,112.235207,92.380021,133.245203,122.334312,112.490701,112.235207,92.380021,...,108.057420,122.334312,112.490701,112.235207,92.380021,57.894915,122.334312,112.490701,112.235207,92.380021
2,abortion_miscarriage_ectopic_pregnancy,122.334312,112.490701,112.235207,92.380021,133.245203,122.334312,112.490701,112.235207,92.380021,...,108.057420,122.334312,112.490701,112.235207,92.380021,57.894915,122.334312,112.490701,112.235207,92.380021
3,mortality,122.334312,112.490701,112.235207,92.380021,133.245203,122.334312,112.490701,112.235207,92.380021,...,108.057420,122.334312,112.490701,112.235207,92.380021,57.894915,122.334312,112.490701,112.235207,92.380021
4,early_neonatal_mortality,122.334312,103.176540,96.089689,67.407822,131.738060,122.334312,103.176540,96.089689,67.407822,...,106.132866,122.334312,103.176540,96.089689,67.407822,51.682077,122.334312,103.176540,96.089689,67.407822
5,late_neonatal_mortality,122.334312,112.490701,112.235207,92.380021,133.245203,122.334312,112.490701,112.235207,92.380021,...,108.057420,122.334312,112.490701,112.235207,92.380021,57.894915,122.334312,112.490701,112.235207,92.380021
6,postpartum_hemoglobin_nine_month,136.866261,120.963279,126.332787,100.372639,147.540470,136.866261,120.963279,126.332787,100.372639,...,122.380718,136.866261,120.963279,126.332787,100.372639,71.134319,136.866261,120.963279,126.332787,100.372639
7,postpartum_depression,122.334312,112.490701,112.235207,92.380021,133.245203,122.334312,112.490701,112.235207,92.380021,...,108.057420,122.334312,112.490701,112.235207,92.380021,57.894915,122.334312,112.490701,112.235207,92.380021


In [73]:
diffs = step_shifts[["next_step"]].copy()

# overall
diffs["aph"] =  step_shifts["aph_no_pph"] - step_shifts["no_aph_no_pph"]
diffs["pph"] =  step_shifts["no_aph_pph"] - step_shifts["no_aph_no_pph"]
diffs["both"] = step_shifts["aph_pph"] -    step_shifts["no_aph_no_pph"]

# not anemic
diffs["not_anemic_aph"] =  step_shifts["not_anemic_aph_no_pph"] - step_shifts["not_anemic_no_aph_no_pph"]
diffs["not_anemic_pph"] =  step_shifts["not_anemic_no_aph_pph"] - step_shifts["not_anemic_no_aph_no_pph"]
diffs["not_anemic_both"] = step_shifts["not_anemic_aph_pph"] -    step_shifts["not_anemic_no_aph_no_pph"]

# mild anemia
diffs["mild_aph"] =  step_shifts["mild_aph_no_pph"] - step_shifts["mild_no_aph_no_pph"]
diffs["mild_pph"] =  step_shifts["mild_no_aph_pph"] - step_shifts["mild_no_aph_no_pph"]
diffs["mild_both"] = step_shifts["mild_aph_pph"] -    step_shifts["mild_no_aph_no_pph"]

# moderate anemia
diffs["moderate_aph"] =  step_shifts["moderate_aph_no_pph"] - step_shifts["moderate_no_aph_no_pph"]
diffs["moderate_pph"] =  step_shifts["moderate_no_aph_pph"] - step_shifts["moderate_no_aph_no_pph"]
diffs["moderate_both"] = step_shifts["moderate_aph_pph"] -    step_shifts["moderate_no_aph_no_pph"]

# severe anemia
diffs["severe_aph"] =  step_shifts["severe_aph_no_pph"] - step_shifts["severe_no_aph_no_pph"]
diffs["severe_pph"] =  step_shifts["severe_no_aph_pph"] - step_shifts["severe_no_aph_no_pph"]
diffs["severe_both"] = step_shifts["severe_aph_pph"] -    step_shifts["severe_no_aph_no_pph"]

diffs

,next_step,aph,pph,both,not_anemic_aph,not_anemic_pph,not_anemic_both,mild_aph,mild_pph,mild_both,moderate_aph,moderate_pph,moderate_both,severe_aph,severe_pph,severe_both
0,maternal_sepsis_and_other_maternal_infections,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292
1,residual_maternal_disorders,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292
2,abortion_miscarriage_ectopic_pregnancy,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292
3,mortality,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292
4,early_neonatal_mortality,-19.157772,-26.244623,-54.926490,-19.157772,-26.244623,-54.926490,-19.157772,-26.244623,-54.926490,-19.157772,-26.244623,-54.926490,-19.157772,-26.244623,-54.926490
5,late_neonatal_mortality,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292
6,postpartum_hemoglobin_nine_month,-15.902981,-10.533474,-36.493622,-15.902981,-10.533474,-36.493622,-15.902981,-10.533474,-36.493622,-15.902981,-10.533474,-36.493622,-15.902981,-10.533474,-36.493622
7,postpartum_depression,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292,-9.843611,-10.099105,-29.954292


In [74]:
diff_deltas = diffs.set_index(["next_step"]).diff()
diff_deltas

,aph,pph,both,not_anemic_aph,not_anemic_pph,not_anemic_both,mild_aph,mild_pph,mild_both,moderate_aph,moderate_pph,moderate_both,severe_aph,severe_pph,severe_both
next_step,,,,,,,,,,,,,,,
maternal_sepsis_and_other_maternal_infections,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
residual_maternal_disorders,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
abortion_miscarriage_ectopic_pregnancy,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
mortality,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
early_neonatal_mortality,-9.314161,-16.145518,-24.972199,-9.314161,-16.145518,-24.972199,-9.314161,-16.145518,-24.972199,-9.314161,-16.145518,-24.972199,-9.314161,-16.145518,-24.972199
late_neonatal_mortality,9.314161,16.145518,24.972199,9.314161,16.145518,24.972199,9.314161,16.145518,24.972199,9.314161,16.145518,24.972199,9.314161,16.145518,24.972199
postpartum_hemoglobin_nine_month,-6.059370,-0.434369,-6.539331,-6.059370,-0.434369,-6.539331,-6.059370,-0.434369,-6.539331,-6.059370,-0.434369,-6.539331,-6.059370,-0.434369,-6.539331
postpartum_depression,6.059370,0.434369,6.539331,6.059370,0.434369,6.539331,6.059370,0.434369,6.539331,6.059370,0.434369,6.539331,6.059370,0.434369,6.539331
